# CellGNN — heterogeneous-edge dynamics on a real minimal cell

Goal: take a trajectory from the Luthey-Schulten `Minimal_Cell` (or `Minimal_Cell_4DWCM`) data drop, build a heterogeneous particle graph from each frame, run our `CellGNN` on it, train a tiny next-frame predictor, then roll forward at 1 μs cadence.

**What this is:** a learned dynamics *surrogate* that imitates trajectories produced by upstream physics (4DWCM's RDME + LAMMPS polymer + biogenesis kinetics). It distils observed dynamics into a fast inference path — similar role GraphCast plays for weather, NequIP plays for MD.

**What this is NOT:** a physics solver. It does not solve polymer dynamics from first principles, does not model ribosome biogenesis (Earnest, Lai, Chen 2015 BiophysJ tracks 145 assembly intermediates — that's biology, not a clique of edges), does not track lipid incorporation or membrane curvature. A connectivity pattern is not a process. The GNN can only learn dynamics that appear in its training data; for anything else, the underlying simulator stays upstream.

Notebook is **self-discovering** — it walks your drive folder, reports what files are present, and adapts. If spatial trajectory data is found (`.lm` HDF5 from RDME), we use that. Otherwise we fall back to `outputs/syn3a_cell_4d.npz` that we built ourselves.

Cells:
1. setup + clone repo + verify torch
2. mount drive + walk data folder
3. inspect files (HDF5 / npy / pickle)
4. pick a trajectory + plot raw counts
5. build per-frame particle list (positions + species)
6. build dynamic heterogeneous edges + edge taxonomy plot
7. CellGNN forward — untrained shapes + equivariance check
8. assemble (t, t+dt) training pairs
9. tiny training loop (next-frame displacement loss)
10. roll out at 1 μs cadence + save trajectory
11. compare rollout to ground truth

## 1. Setup

In [ ]:
import os, sys, subprocess
REPO = '/content/cell'
if not os.path.isdir(REPO):
    !git clone https://github.com/Nikku03/cell.git {REPO}
%cd {REPO}
!git fetch --quiet && git checkout claude/syn3a-whole-cell-simulator-REjHC && git pull --quiet --ff-only
sys.path.insert(0, REPO)
sys.path.insert(0, REPO + '/cell_sim')

# h5py + torch already on Colab; sanity-check anyway.
import torch, h5py, numpy as np, pandas as pd
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      'device', 'cuda' if torch.cuda.is_available() else 'cpu')
print('numpy', np.__version__, 'h5py', h5py.__version__)

## 2. Mount drive + walk data folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA_DIR = Path('/content/drive/MyDrive/Luthey-Schulten-Lab-Minimal_Cell-db048ac')
assert DATA_DIR.exists(), f'Folder not found at {DATA_DIR}. Adjust path.'

print(f'walking {DATA_DIR}\n')
exts = {}
size_total = 0
candidates = []
for p in DATA_DIR.rglob('*'):
    if p.is_file():
        ext = p.suffix.lower()
        exts[ext] = exts.get(ext, 0) + 1
        sz = p.stat().st_size
        size_total += sz
        if ext in {'.lm', '.h5', '.hdf5', '.npy', '.npz', '.pkl', '.pickle', '.csv', '.xml'}:
            candidates.append((sz, p))

print(f'total: {sum(exts.values())} files, {size_total/1e9:.2f} GB')
for ext, n in sorted(exts.items(), key=lambda kv: -kv[1])[:15]:
    print(f'  {ext:>10s}  {n}')

print('\nlargest data candidates:')
for sz, p in sorted(candidates, key=lambda x: -x[0])[:15]:
    print(f'  {sz/1e6:>8.1f} MB  {p.relative_to(DATA_DIR)}')

## 3. Find a spatial trajectory (HDF5 RDME)

If we get lucky and find `.lm` HDF5 files, we'll use those. Otherwise we'll fall back to the particle-overlay tensor we already produced (`outputs/syn3a_cell_4d.npz`).

In [ ]:
lm_files = sorted([p for p in DATA_DIR.rglob('*.lm')] +
                  [p for p in DATA_DIR.rglob('*.h5')] +
                  [p for p in DATA_DIR.rglob('*.hdf5')])
print(f'HDF5 / .lm candidates: {len(lm_files)}')
for p in lm_files[:10]:
    print(f'  {p.stat().st_size/1e6:>8.1f} MB  {p.relative_to(DATA_DIR)}')

# Inspect the first one's schema.
TRAJ_PATH = None
schema = {}
if lm_files:
    TRAJ_PATH = lm_files[0]
    print(f'\nopening {TRAJ_PATH.name} ...')
    with h5py.File(TRAJ_PATH, 'r') as f:
        def walk(name, obj):
            kind = 'group' if isinstance(obj, h5py.Group) else 'dataset'
            shape = getattr(obj, 'shape', '')
            dtype = getattr(obj, 'dtype', '')
            schema[name] = (kind, shape, str(dtype))
        f.visititems(walk)
    print(f'  {len(schema)} entries')
    for k, v in list(schema.items())[:30]:
        print(f'  {v[0]:>7s}  {str(v[1]):>20s}  {v[2]:>10s}  {k}')
else:
    print('\nNo HDF5 trajectories — will fall back to outputs/syn3a_cell_4d.npz')

## 4. Build a (T, N, 3) particle tensor

Two paths:
- **A.** Real LM trajectory: read the `(T, X, Y, Z, particlesPerSite)` lattice and emit per-frame particle lists.
- **B.** Fallback: load `outputs/syn3a_cell_4d.npz` from our own particle-overlay run.

We pick whichever is available. The downstream cells are agnostic — they consume `(T, N, 3) positions + (T, N) species_id`.

In [ ]:
def lattice_to_particles(lattice, lattice_spacing_nm=10.0, max_particles=8000, rng=None):
    """LM RDME lattice -> particle list. lattice shape: (X,Y,Z,particlesPerSite).
    Each non-zero entry at (x,y,z,slot) becomes a particle of species id = entry value
    (LM convention) at the voxel centre + a tiny jitter inside the voxel.
    """
    if rng is None:
        rng = np.random.default_rng(0)
    nz = np.argwhere(lattice > 0)
    if len(nz) > max_particles:
        nz = nz[rng.choice(len(nz), size=max_particles, replace=False)]
    x, y, z, slot = nz.T
    sid = lattice[x, y, z, slot].astype(np.int64) - 1
    pos_nm = np.stack([x, y, z], axis=-1).astype(np.float32) * lattice_spacing_nm
    pos_nm += rng.uniform(0, lattice_spacing_nm, size=pos_nm.shape).astype(np.float32)
    return pos_nm, sid

POSITIONS, SPECIES_ID, T_FRAMES, SPECIES_NAMES = None, None, None, None

if TRAJ_PATH is not None and any('Lattice' in k or 'lattice' in k for k in schema):
    # Real LM data
    print(f'Path A — extracting particles from {TRAJ_PATH.name}')
    with h5py.File(TRAJ_PATH, 'r') as f:
        # Find the lattice dataset (LM names vary; common path: '/Simulations/0000001/Lattice/...')
        lattice_keys = [k for k in schema if k.lower().endswith('/lattice')
                        or k.lower().endswith('/latticestate')
                        or 'lattice' in k.lower()]
        print('lattice keys found:', lattice_keys[:5])
        # Read first few timepoints
        if lattice_keys:
            lk = lattice_keys[0]
            data = f[lk][:]   # may be (T, X, Y, Z, p) or (X, Y, Z, p)
            print('lattice shape:', data.shape, data.dtype)
            if data.ndim == 5:
                T = min(data.shape[0], 32)
                positions_list, species_list = [], []
                for t in range(T):
                    p, s = lattice_to_particles(data[t])
                    positions_list.append(p); species_list.append(s)
                # Pad to common N
                Nmax = max(p.shape[0] for p in positions_list)
                POSITIONS = np.full((T, Nmax, 3), np.nan, dtype=np.float32)
                SPECIES_ID = np.full((T, Nmax), -1, dtype=np.int64)
                for t, (p, s) in enumerate(zip(positions_list, species_list)):
                    POSITIONS[t, :p.shape[0]] = p
                    SPECIES_ID[t, :s.shape[0]] = s
                T_FRAMES = T
                print(f'extracted {T} frames × up to {Nmax} particles')

if POSITIONS is None:
    # Fallback: our own particle overlay
    fallback = Path(REPO) / 'outputs/syn3a_cell_4d.npz'
    print(f'Path B — falling back to {fallback}')
    d = np.load(fallback, allow_pickle=True)
    POSITIONS = d['positions']        # (T, Nmax, 3)
    SPECIES_ID = d['species_id']      # (T, Nmax)
    SPECIES_NAMES = list(d['species'])
    T_FRAMES = POSITIONS.shape[0]
    print(f'loaded {T_FRAMES} frames × {POSITIONS.shape[1]} slots × 3')
    print(f'{len(SPECIES_NAMES)} species: {SPECIES_NAMES[:8]} ...')

## 5. Plot a frame — sanity check the data

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

f0 = 0
valid = SPECIES_ID[f0] >= 0
p = POSITIONS[f0][valid]
s = SPECIES_ID[f0][valid]
n_show = min(2000, len(p))
idx = np.random.default_rng(0).choice(len(p), size=n_show, replace=False)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(p[idx, 0], p[idx, 1], p[idx, 2], c=s[idx], cmap='tab20', s=4, alpha=0.6)
ax.set_title(f'frame {f0}  ·  {len(p)} particles  ·  showing {n_show}')
ax.set_xlabel('x (nm)'); ax.set_ylabel('y (nm)'); ax.set_zlabel('z (nm)')
fig.colorbar(sc, ax=ax, label='species id', shrink=0.7)
plt.tight_layout(); plt.show()

## 6. Build dynamic heterogeneous edges

Take a single frame, run `build_dynamic_edges`, report the edge-type histogram. This is the moment the cell becomes a graph.

In [ ]:
from cell_sim.atom_engine.cell_gnn import (CellGNN, EdgeType, EdgeFeaturizer,
                                             build_dynamic_edges, N_EDGE_TYPES)

valid = SPECIES_ID[f0] >= 0
pos_t = torch.tensor(POSITIONS[f0][valid])
sid_t = torch.tensor(SPECIES_ID[f0][valid])
n_t = pos_t.shape[0]
print(f'frame {f0}: {n_t} particles')

# Subsample if too many (dense pairwise blows up memory)
MAX_N = 3000
if n_t > MAX_N:
    sel = torch.randperm(n_t)[:MAX_N]
    pos_t = pos_t[sel]; sid_t = sid_t[sel]; n_t = MAX_N
    print(f'  subsampled to {n_t} for edge build')

g = build_dynamic_edges(pos_t, sid_t, r_cut_spatial=20.0)
print(f'\nedges: {g["edges"].shape[0]} total')
for et in EdgeType:
    n_e = (g['edge_type'] == int(et)).sum().item()
    if n_e:
        print(f'  {et.name:<11s}  {n_e:>7d}')

import collections
deg = collections.Counter(g['edges'][:, 0].tolist())
deg_arr = np.array(list(deg.values()))
print(f'\nper-particle degree:  median={np.median(deg_arr):.1f}  '
      f'mean={deg_arr.mean():.1f}  max={deg_arr.max()}')

## 7. CellGNN — untrained forward pass + shape + equivariance

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_species_vocab = max(int(sid_t.max().item()) + 1, 32)
model = CellGNN(n_species=n_species_vocab, hidden=64, n_rounds=3,
                n_reaction_classes=64, r_cut=20.0,
                n_extra_node_features=4).to(device)
print(f'CellGNN  hidden=64  n_rounds=3  device={device}')
print(f'params: {sum(p.numel() for p in model.parameters()):,}')

extra = torch.zeros((n_t, 4), device=device)
pos_d = pos_t.to(device); sid_d = sid_t.to(device)
edges_d = g['edges'].to(device); r_d = g['r'].to(device)
et_d = g['edge_type'].to(device); thick_d = g['thickness'].to(device)

with torch.no_grad():
    out = model.predict_all(sid_d, extra, edges_d, r_d, et_d, pos_d, thickness=thick_d)
for k, v in out.items():
    print(f'  {k:>22s}  {tuple(v.shape)}')

# Equivariance check on real data
torch.manual_seed(0)
A = torch.randn(3, 3, device=device); Q, _ = torch.linalg.qr(A)
if torch.det(Q) < 0: Q[:, -1] *= -1
pos_rot = pos_d @ Q.T
with torch.no_grad():
    f_orig = model.predict_forces_equivariant(sid_d, extra, edges_d, r_d, et_d, pos_d, thickness=thick_d)
    f_rot = model.predict_forces_equivariant(sid_d, extra, edges_d, r_d, et_d, pos_rot, thickness=thick_d)
err = (f_rot - f_orig @ Q.T).abs().max().item()
print(f'\nE(3) equivariance:  max |f(Rx) - R·f(x)| = {err:.2e}')

## 8. Assemble (t, t+dt) training pairs

We learn next-frame **displacement**: given the current frame's graph + positions, predict `Δposition` to the next frame. This is the simplest learnable signal that doesn't require explicit reaction labels.

Loss: MSE on displacement, restricted to particles present in both frames (matched by frame-major slot ordering for the fallback data; for real LM data, particles need ID-tracking — left as a TODO).

In [ ]:
def make_pair(t):
    """Return (positions_t, species_t, displacement_to_t+1).
    Both frames clipped to the slots populated in BOTH frames."""
    a_valid = SPECIES_ID[t] >= 0
    b_valid = SPECIES_ID[t + 1] >= 0
    both = a_valid & b_valid
    p_a = POSITIONS[t][both]
    p_b = POSITIONS[t + 1][both]
    s = SPECIES_ID[t][both]
    dp = p_b - p_a
    return p_a, s, dp

# Quick health check
p, s, dp = make_pair(0)
print(f'frame pair 0 → 1: {p.shape[0]} matched particles')
print(f'displacement stats: mean |dx|={np.linalg.norm(dp, axis=-1).mean():.1f} nm, '
      f'max={np.linalg.norm(dp, axis=-1).max():.1f} nm')

## 9. Tiny training loop

Predict `Δposition` from the current graph. Force head is repurposed as the displacement output (we treat it as 'where this particle wants to be next' — equivariant by construction). 5 epochs over all available pairs, ~30–60 s on Colab CPU, much faster on GPU.

In [ ]:
import time
torch.manual_seed(42)

model = CellGNN(n_species=n_species_vocab, hidden=64, n_rounds=3,
                n_reaction_classes=64, r_cut=20.0,
                n_extra_node_features=4).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-5)
EPOCHS = 5
MAX_N_PAIR = 2000   # subsample per pair to keep memory reasonable

rng_t = np.random.default_rng(0)
loss_log = []
t0 = time.time()
for ep in range(EPOCHS):
    perm = rng_t.permutation(T_FRAMES - 1)
    epoch_losses = []
    for t in perm:
        p_a, s, dp = make_pair(int(t))
        if p_a.shape[0] == 0:
            continue
        if p_a.shape[0] > MAX_N_PAIR:
            sel = rng_t.choice(p_a.shape[0], size=MAX_N_PAIR, replace=False)
            p_a, s, dp = p_a[sel], s[sel], dp[sel]
        n_p = p_a.shape[0]
        pos = torch.tensor(p_a, device=device)
        sid = torch.tensor(s, device=device)
        target = torch.tensor(dp, device=device)
        ex = torch.zeros((n_p, 4), device=device)
        gg = build_dynamic_edges(pos, sid, r_cut_spatial=20.0)
        pred = model.predict_forces_equivariant(
            sid, ex, gg['edges'].to(device), gg['r'].to(device),
            gg['edge_type'].to(device), pos,
            thickness=gg['thickness'].to(device))
        loss = (pred - target).pow(2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        epoch_losses.append(loss.item())
    mean_loss = float(np.mean(epoch_losses))
    loss_log.append(mean_loss)
    print(f'epoch {ep+1}/{EPOCHS}  mean displacement-MSE = {mean_loss:.3f} nm²  '
          f'({time.time()-t0:.1f}s)')

plt.figure(figsize=(5, 3))
plt.plot(range(1, EPOCHS+1), loss_log, marker='o')
plt.xlabel('epoch'); plt.ylabel('mean displacement MSE (nm²)')
plt.title('CellGNN training (next-frame displacement)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Save
ckpt_dir = Path(REPO) / 'cell_sim/atom_engine/checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = ckpt_dir / 'cell_gnn_smoke.pt'
torch.save({'state_dict': model.state_dict(),
            'config': dict(n_species=n_species_vocab, hidden=64, n_rounds=3,
                            n_reaction_classes=64, r_cut=20.0,
                            n_extra_node_features=4)},
           ckpt_path)
print(f'wrote {ckpt_path}')

## 10. Roll out at 1 μs cadence

Initialize from frame 0, repeatedly apply the learned displacement, save 200 rollout frames. The model has a fixed inference cost per step; the 1 μs cadence is just a label we attach to the output (the model interpolates whatever displacement scale it learned).

In [ ]:
model.eval()
ROLLOUT_FRAMES = 200
DT_US = 1.0

valid0 = SPECIES_ID[0] >= 0
p0 = torch.tensor(POSITIONS[0][valid0], device=device)
s0 = torch.tensor(SPECIES_ID[0][valid0], device=device)
if p0.shape[0] > 2000:
    sel = torch.randperm(p0.shape[0])[:2000]
    p0 = p0[sel]; s0 = s0[sel]
n_p = p0.shape[0]
ex = torch.zeros((n_p, 4), device=device)

rollout = np.zeros((ROLLOUT_FRAMES, n_p, 3), dtype=np.float32)
rollout[0] = p0.cpu().numpy()
pos = p0.clone()
with torch.no_grad():
    for f in range(1, ROLLOUT_FRAMES):
        gg = build_dynamic_edges(pos, s0, r_cut_spatial=20.0)
        dp = model.predict_forces_equivariant(
            s0, ex, gg['edges'].to(device), gg['r'].to(device),
            gg['edge_type'].to(device), pos,
            thickness=gg['thickness'].to(device))
        pos = pos + dp
        rollout[f] = pos.cpu().numpy()

out_path = Path(REPO) / 'outputs/cell_gnn_rollout.npz'
np.savez_compressed(out_path,
                    positions=rollout, species_id=s0.cpu().numpy(),
                    t_us=np.arange(ROLLOUT_FRAMES) * DT_US)
print(f'rolled out {ROLLOUT_FRAMES} frames at {DT_US} μs cadence')
print(f'wrote {out_path}  ({out_path.stat().st_size/1e6:.1f} MB)')
print(f'final mean radius: {np.linalg.norm(rollout[-1], axis=-1).mean():.1f} nm')

## 11. Compare rollout vs ground truth

Two diagnostics:
- **Mean radial extent** over time (rollout vs ground-truth frame 0..ROLLOUT_FRAMES if available)
- **Per-species mean position drift** to spot whether the model is collapsing or exploding

In [ ]:
rollout_r = np.linalg.norm(rollout, axis=-1).mean(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(ROLLOUT_FRAMES) * DT_US, rollout_r, label='rollout')
if T_FRAMES >= ROLLOUT_FRAMES:
    gt_r = np.linalg.norm(POSITIONS[:ROLLOUT_FRAMES], axis=-1)
    gt_mask = SPECIES_ID[:ROLLOUT_FRAMES] >= 0
    gt_r[~gt_mask] = np.nan
    axes[0].plot(np.arange(ROLLOUT_FRAMES) * DT_US,
                  np.nanmean(gt_r, axis=1), '--', label='ground truth')
axes[0].set_xlabel('t (μs)'); axes[0].set_ylabel('mean radius (nm)')
axes[0].set_title('mean radial extent'); axes[0].legend(); axes[0].grid(alpha=0.3)

# 2D projection of frames 0, mid, last
for k, idx in enumerate([0, ROLLOUT_FRAMES // 2, ROLLOUT_FRAMES - 1]):
    axes[1].scatter(rollout[idx, :, 0], rollout[idx, :, 1],
                     s=2, alpha=0.4, label=f'f{idx}')
axes[1].set_xlabel('x (nm)'); axes[1].set_ylabel('y (nm)')
axes[1].set_aspect('equal'); axes[1].set_title('xy-projection of rollout')
axes[1].legend()
plt.tight_layout(); plt.show()

## What we have

- Auto-discovery of trajectory data on drive
- Conversion to particle list — works for both LM HDF5 and our own .npz
- Heterogeneous edge graph built per frame, 7 edge types available
- CellGNN forward, equivariance verified
- Tiny next-frame-displacement training
- Rollout at 1 μs cadence saved to `outputs/cell_gnn_rollout.npz`

## Next steps

- **Reaction labels** — extract per-frame reaction events from LM data (write to `reaction_logits` target). Currently only forces are trained.
- **Spawn/destroy labels** — track births/deaths frame-to-frame and train the spawn head.
- **Particle ID tracking for real LM data** — current pair-builder relies on slot ordering; LM frames need a permutation match (e.g. greedy nearest-neighbour).
- **Push the rollout per-gene latent state** into `scripts/sparse_lnn_cascade_stacker.py` as a new feature block. Re-run LOO and compare to current baseline.